# 03. 그래프 분류

앞 노트북은 한 그래프 안의 노드를 분류했습니다. 이번에는 **그래프 하나가 하나의 샘플**인 그래프 분류를 다룹니다
(예: 분자 그래프 → 독성 여부, 단백질 구조 → 기능 분류).

개념을 명확히 보기 위해 두 종류의 그래프(촘촘한 그래프 vs 듬성한 그래프)를 만들어 분류합니다.

1. 그래프 데이터셋 생성
2. 그래프 단위 GCN + 풀링
3. 배치 학습 및 평가

In [ ]:
import torch
import torch.nn.functional as F
import networkx as nx
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. 그래프 데이터셋 생성

두 클래스의 그래프를 만듭니다.
- 클래스 0: 촘촘한 그래프 (연결 확률 높음)
- 클래스 1: 듬성한 그래프 (연결 확률 낮음)

각 노드에는 차수(degree) 기반 특징을 줍니다.

In [ ]:
def make_graph(label, rng):
    n = rng.integers(15, 25)
    p = 0.4 if label == 0 else 0.12       # 클래스별 연결 확률
    g = nx.erdos_renyi_graph(n, p, seed=int(rng.integers(1e6)))
    if g.number_of_edges() == 0:
        g.add_edge(0, 1)
    data = from_networkx(g)
    deg = torch.tensor([d for _, d in g.degree()], dtype=torch.float).view(-1, 1)
    data.x = deg                            # 노드 특징 = 차수
    data.y = torch.tensor([label])
    return data

import numpy as np
rng = np.random.default_rng(0)
graphs = [make_graph(i % 2, rng) for i in range(200)]
train_ds, test_ds = graphs[:160], graphs[160:]

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)
print('학습 그래프:', len(train_ds), '| 테스트 그래프:', len(test_ds))

## 2. 그래프 단위 GCN + 풀링

노드별 GCN 출력을 `global_mean_pool`로 **그래프 하나의 벡터**로 모은 뒤 분류합니다.
배치 내 여러 그래프는 `batch` 인덱스로 구분됩니다.

In [ ]:
class GraphGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden, n_classes):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = torch.nn.Linear(hidden, n_classes)
    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)      # 그래프별 평균 -> (그래프 수, hidden)
        return self.lin(x)

model = GraphGCN(1, 32, 2).to(device)
print(model)

## 3. 배치 학습 및 평가

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def evaluate(loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch).argmax(1)
            correct += int((pred == batch.y).sum()); total += batch.num_graphs
    return correct / total

for epoch in range(30):
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward(); optimizer.step()

print(f'테스트 정확도: {evaluate(test_loader)*100:.1f}%')

### 정리

- 그래프 하나를 하나의 샘플로 보는 그래프 분류를 구현했습니다.
- `global_mean_pool`로 노드 표현을 그래프 표현으로 모으고, 배치 학습으로 분류했습니다.
- 실제로는 분자(MUTAG·ENZYMES 등), 단백질, 소셜 네트워크 그래프 데이터셋에 같은 방식을 적용합니다.